In [1]:
# Install Dependencies
!pip install polyglot
!pip install pyicu
!pip install pycld2
!pip install morfessor
!pip install wordcloud
!pip install seaborn
!pip install stanza

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

# Install Polyglot Embeddings untuk Bahasa Indonesia
!polyglot download pos2.id
!polyglot download embeddings2.id


  Using cached pyicu-2.15.3.tar.gz (267 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
  Using cached pyicu-2.15.3.tar.gz (267 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [66 lines of output]
      (running 'icu-config --version')
      (running 'pkg-config --modversion icu-i18n')
      Traceback (most recent call last):
        File "<string>", line 89, in <module>
        File "<frozen os>", line 679, in __getitem__
      KeyError: 'ICU_VERSION'
      
      During handling of the above exception, another exception occurred:
      
      Traceback (most recent call last):
        File "<string>", line 92, in <module>
        File "<string>", line 19, in check_output
        File "C:\Users\Anas G\anaconda3\Lib\subprocess.py", line 466, in check_output
          return run(*popenargs, stdout=PIPE, timeout=timeout, check=True,
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\Anas G\anaconda3\Lib\subprocess.py", line 548, in run
          with Popen(*popenargs, **kwargs) as proce

[nltk_data] Downloading package punkt to C:\Users\Anas
[nltk_data]     G\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Anas
[nltk_data]     G\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Anas G\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Anas G\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Anas G\anaconda3\Scripts\polyglot.exe\__main__.py", line 4, in <module>
  File "C:\Users\Anas

In [2]:
import pandas as pd

input_filepath = '../Week2/df_tj_sentiment_analysis.csv' 
try:
    df_tj = pd.read_csv(input_filepath)
    print(f"File '{input_filepath}' berhasil dibaca. Jumlah data: {len(df_tj)} baris.")
except FileNotFoundError:
    print(f"Error: File '{input_filepath}' tidak ditemukan.")
    exit()

File '../Week2/df_tj_sentiment_analysis.csv' berhasil dibaca. Jumlah data: 3016 baris.


In [3]:
import re
import pandas as pd
from textblob import TextBlob
from collections import Counter
from IPython.display import display

# 1. Define mapping of POS codes to human‑friendly names
pos_label_map = {
    'NN': 'Noun, singular or mass',
    'NNS': 'Noun, plural',
    'NNP': 'Proper noun, singular',
    'NNPS': 'Proper noun, plural',
    'VB': 'Verb, base form',
    'VBD': 'Verb, past tense',
    'VBG': 'Verb, gerund/present participle',
    'VBN': 'Verb, past participle',
    'VBP': 'Verb, non‑3rd person singular present',
    'VBZ': 'Verb, 3rd person singular present',
    'JJ': 'Adjective',
    'JJR': 'Adjective, comparative',
    'JJS': 'Adjective, superlative',
    'RB': 'Adverb',
    'RBR': 'Adverb, comparative',
    'RBS': 'Adverb, superlative',
    'PRP': 'Personal pronoun',
    'PRP$': 'Possessive pronoun',
    'IN': 'Preposition/subordinating conjunction',
    'DT': 'Determiner',
    'CC': 'Coordinating conjunction',
    'UH': 'Interjection',
    'CD': 'Cardinal number',
    'EX': 'Existential there',
    'FW': 'Foreign word',
    'LS': 'List item marker',
    'MD': 'Modal',
    'PDT': 'Predeterminer',
    'POS': 'Possessive ending',
    'RP': 'Particle',
    'SYM': 'Symbol',
    'TO': 'to',
    'UH': 'Interjection',
    'WDT': 'Wh-determiner',
    'WP': 'Wh-pronoun',
    'WP$': 'Possessive wh-pronoun',
    'WRB': 'Wh-adverb'
    # Add more tags if needed
}

# 2. Collect all POS tags from each cleaned review
all_pos = []
word_pos_map = {}
for content in df_tj['cleaned_content_no_common'].fillna(''):
    if content.strip():
        for word, pos in TextBlob(content).tags:
            all_pos.append(pos)
            word_pos_map.setdefault(pos, []).append(word.lower())

# 3. Compute overall counts and unique token counts per tag
pos_counts = Counter(all_pos)
unique_counts = {pos: len(set(words)) for pos, words in word_pos_map.items()}

# 4. Build DataFrame
rows = []
for pos, count in pos_counts.items():
    tag_name = pos_label_map.get(pos)
    if tag_name is None:
        tag_name = 'Other (Tag belum terdefinisi di POS map)'
    rows.append({
        'Tag': pos,
        'Tag Name': tag_name,
        'Count': count,
        'Unique Tokens': unique_counts.get(pos, 0)
    })

pos_df = pd.DataFrame(rows)

# 5. Sort by Count descending and display as table
pos_df = pos_df.sort_values('Count', ascending=False).reset_index(drop=True)
display(pos_df)


,Tag,Tag Name,Count,Unique Tokens
0,NN,"Noun, singular or mass",20366,2400
1,JJ,Adjective,2304,565
2,VBP,"Verb, non‑3rd person singular present",530,201
3,VBD,"Verb, past tense",366,162
4,CD,Cardinal number,342,115
5,NNS,"Noun, plural",326,105
6,IN,Preposition/subordinating conjunction,324,98
7,VBZ,"Verb, 3rd person singular present",146,90
8,FW,Foreign word,136,51
9,VB,"Verb, base form",107,51


In [4]:
# Filter data frame untuk sentimen negatif
df_negatif = df_tj[df_tj['sentiment_rating'] == 'Negative'].copy()

# Pastikan kolom 'stopword_removal' terisi (handle NaN)
df_negatif['cleaned_content_no_common'] = df_negatif['cleaned_content_no_common'].fillna('')

# Kumpulkan semua POS tags dari review dengan sentimen negatif
all_pos_negatif = []
word_pos_map_negatif = {}

for content in df_negatif['cleaned_content_no_common']:
    if content.strip(): # Pastikan konten tidak kosong setelah fillna
        for word, pos in TextBlob(content).tags:
            all_pos_negatif.append(pos)
            # Mengumpulkan kata-kata unik untuk setiap tag POS
            word_pos_map_negatif.setdefault(pos, set()).add(word.lower())


# Hitung frekuensi masing-masing tag POS
pos_counts_negatif = Counter(all_pos_negatif)

# Hitung jumlah token unik per tag
unique_counts_negatif = {pos: len(words) for pos, words in word_pos_map_negatif.items()}

# Buat DataFrame untuk hasil POS tagging sentimen negatif
rows_negatif = []
for pos, count in pos_counts_negatif.items():
    tag_name = pos_label_map.get(pos) # Gunakan mapping yang sudah ada
    if tag_name is None:
        tag_name = 'Other (Tag belum terdefinisi di POS map)'
    rows_negatif.append({
        'Tag': pos,
        'Tag Name': tag_name,
        'Count': count,
        'Unique Tokens': unique_counts_negatif.get(pos, 0)
    })

pos_df_negatif = pd.DataFrame(rows_negatif)

# Urutkan berdasarkan hitungan (descending)
pos_df_negatif = pos_df_negatif.sort_values('Count', ascending=False).reset_index(drop=True)

# Tampilkan hasil
print("\n=== POS Tagging Results for Negative Sentiment ===")
display(pos_df_negatif)

# Opsional: Tampilkan contoh kata untuk tag POS tertentu pada sentimen negatif
print("\n=== Example Words per POS Tag (Negative Sentiment) ===")
for pos, words in list(word_pos_map_negatif.items())[:10]: # Tampilkan untuk 10 tag pertama
    print(f"Tag '{pos}' ({pos_label_map.get(pos, 'Unknown')}): {list(words)[:10]}...") # Tampilkan hingga 10 kata unik



=== POS Tagging Results for Negative Sentiment ===


,Tag,Tag Name,Count,Unique Tokens
0,NN,"Noun, singular or mass",4336,1243
1,JJ,Adjective,708,284
2,CD,Cardinal number,238,89
3,VBD,"Verb, past tense",128,92
4,VBP,"Verb, non‑3rd person singular present",91,61
5,NNS,"Noun, plural",90,44
6,IN,Preposition/subordinating conjunction,86,41
7,RB,Adverb,46,25
8,VB,"Verb, base form",46,23
9,VBZ,"Verb, 3rd person singular present",35,32



=== Example Words per POS Tag (Negative Sentiment) ===
Tag 'NN' (Noun, singular or mass): ['bayar', 'checker', 'kereta', 'ribu', 'kyknya', 'sirih', 'menu', 'lu', 'profesion', 'jagung']...
Tag 'FW' (Foreign word): ['kalo', 'bikin', 'modifikasi', 'ovo', 'msk', 'minggu', 'elektronik', 'bkin', 'karna', 'mngkin']...
Tag 'IN' (Preposition/subordinating conjunction): ['tau', 'osaka', 'tunjukin', 'ngeliat', 'belah', 'nentuin', 'lihat', 'in', 'akurat', 'obat']...
Tag 'VBP' (Verb, non‑3rd person singular present): ['brp', 'tj', 'rute', 'reken', 'nama', 'fitur', 'kalo', 'barat', 'mah', 'make']...
Tag 'JJ' (Adjective): ['follow', 'salah2', 'rute', 'bayar', 'rincian', 'solusi', 'tau', 'otomati', 'ujan', 'amsyooong']...
Tag 'VBD' (Verb, past tense): ['sat', 'anj', 'larang', 'ulang', 'aplikasih', 'bu', 'blm', 'kartu', 'saran', 'sih']...
Tag 'CD' (Cardinal number): ['14a', '27', '120', '6h', '11', '0', '15', '5jam', '5mnt', '45']...
Tag 'JJR' (Adjective, comparative): ['super', 'bener']...
Tag 'VBZ' 

## Compute POS Tag Counts & Counting Common Tagged Words

In [8]:
import stanza
import pandas as pd
from collections import Counter
import os
import tempfile

# Inisialisasi pipeline Bahasa Indonesia
stanza.download('id')  # hanya perlu sekali
nlp = stanza.Pipeline('id', processors='tokenize,pos')

# Fungsi tagging (tahan terhadap input bukan-string / NaN)
def get_pos_tags(text):
    if text is None:
        return []
    if not isinstance(text, str):
        try:
            text = str(text)
        except Exception:
            return []
    text = text.strip()
    if text == '':
        return []
    try:
        doc = nlp(text)
    except AssertionError:
        return []
    except Exception as e:
        return []
    return [(word.text.lower(), word.upos) for sentence in doc.sentences for word in sentence.words]

# POS tagging untuk seluruh data
df_tj['POS'] = df_tj['cleaned_content_no_common'].apply(get_pos_tags)

# DEBUG: Cek nilai unik di kolom sentiment_rating
print("=== DEBUG INFO ===")
print(f"Total rows in df_tj: {len(df_tj)}")
print(f"\nUnique values in 'sentiment_rating' column:")
print(df_tj['sentiment_rating'].value_counts())
print(f"\nData types: {df_tj['sentiment_rating'].dtype}")

# Normalisasi kolom sentiment_rating (strip whitespace dan lowercase)
df_tj['sentiment_rating'] = df_tj['sentiment_rating'].astype(str).str.strip().str.lower()

print(f"\nAfter normalization:")
print(df_tj['sentiment_rating'].value_counts())

# Filter berdasarkan sentimen
df_pos = df_tj[df_tj['sentiment_rating'] == 'positive']
df_neg = df_tj[df_tj['sentiment_rating'] == 'negative']

print(f"\nPositive sentiment rows: {len(df_pos)}")
print(f"Negative sentiment rows: {len(df_neg)}")

# Cek apakah ada data POS yang ter-generate
pos_with_tags = df_pos[df_pos['POS'].apply(lambda x: len(x) > 0)]
neg_with_tags = df_neg[df_neg['POS'].apply(lambda x: len(x) > 0)]
print(f"Positive rows with POS tags: {len(pos_with_tags)}")
print(f"Negative rows with POS tags: {len(neg_with_tags)}")
print("==================\n")

# Fungsi untuk ekstraksi top POS
def extract_top_pos(df_subset):
    pos_counter = Counter()
    word_by_pos = {}

    for pos_tags in df_subset['POS']:
        for word, pos in pos_tags:
            pos_counter[pos] += 1
            word_by_pos.setdefault(pos, []).append(word)

    # Ambil 3 POS paling sering
    top_pos = pos_counter.most_common(3)

    result = []
    for pos, count in top_pos:
        words = word_by_pos[pos]
        common_words = Counter(words).most_common(5)
        result.append({
            'POS': pos,
            'Count': count,
            'Top Words': [w for w, _ in common_words]
        })

    return result

# Ambil hasil top POS
top_pos_pos = extract_top_pos(df_pos)
top_pos_neg = extract_top_pos(df_neg)

print(f"Top POS Positive results: {len(top_pos_pos)} entries")
print(f"Top POS Negative results: {len(top_pos_neg)} entries")

# Buat DataFrame dari hasil tersebut
if top_pos_pos:
    df_top_pos_pos = pd.DataFrame([
        {'POS': entry['POS'], 'Jumlah': entry['Count'], **{f'Kata ke-{i+1}': word for i, word in enumerate(entry['Top Words'])}}
        for entry in top_pos_pos
    ])
else:
    df_top_pos_pos = pd.DataFrame(columns=['POS', 'Jumlah'])
    print("WARNING: No positive POS data found!")

if top_pos_neg:
    df_top_pos_neg = pd.DataFrame([
        {'POS': entry['POS'], 'Jumlah': entry['Count'], **{f'Kata ke-{i+1}': word for i, word in enumerate(entry['Top Words'])}}
        for entry in top_pos_neg
    ])
else:
    df_top_pos_neg = pd.DataFrame(columns=['POS', 'Jumlah'])
    print("WARNING: No negative POS data found!")

# Semua pasangan kata dan POS unik
all_pos_pairs = [pair for pos_list in df_tj['POS'] for pair in pos_list]
unique_pos_pairs = list(set(all_pos_pairs))
df_unique_pos = pd.DataFrame(unique_pos_pairs, columns=['Kata', 'POS']).sort_values(by='Kata')

print(f"Total unique word-POS pairs: {len(df_unique_pos)}")

# Simpan ke Excel (multi-sheet)
default_filename = 'hasil_pos_tag_results_transjakarta.xlsx'
output_path = os.path.join(os.getcwd(), default_filename)

try:
    with pd.ExcelWriter(output_path) as writer:
        df_top_pos_pos.to_excel(writer, index=False, sheet_name='Top POS Positif')
        df_top_pos_neg.to_excel(writer, index=False, sheet_name='Top POS Negatif')
        df_unique_pos.to_excel(writer, index=False, sheet_name='All Words')
    print(f"\nFile '{output_path}' berhasil dibuat dengan 3 sheet:")
    print("- Top POS Positif")
    print("- Top POS Negatif")
    print("- All Words")
except PermissionError:
    temp_dir = tempfile.gettempdir()
    fallback_path = os.path.join(temp_dir, default_filename)
    with pd.ExcelWriter(fallback_path) as writer:
        df_top_pos_pos.to_excel(writer, index=False, sheet_name='Top POS Positif')
        df_top_pos_neg.to_excel(writer, index=False, sheet_name='Top POS Negatif')
        df_unique_pos.to_excel(writer, index=False, sheet_name='All Words')
    print(f"Permission denied writing to '{output_path}'. File ditulis ke '{fallback_path}' sebagai gantinya.")

2025-10-18 11:46:17 INFO: Downloaded file to C:\Users\Anas G\stanza_resources\resources.json
2025-10-18 11:46:17 INFO: Downloading default packages for language: id (Indonesian) ...
2025-10-18 11:46:17 INFO: Downloading default packages for language: id (Indonesian) ...
2025-10-18 11:46:19 INFO: File exists: C:\Users\Anas G\stanza_resources\id\default.zip
2025-10-18 11:46:19 INFO: File exists: C:\Users\Anas G\stanza_resources\id\default.zip
2025-10-18 11:46:21 INFO: Finished downloading models and saved to C:\Users\Anas G\stanza_resources
2025-10-18 11:46:21 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-10-18 11:46:21 INFO: Finished downloading models and saved to C:\Users\Anas G\stanza_resources
2025-10-18 11:46:21 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off wit

2025-10-18 11:46:22 INFO: Downloaded file to C:\Users\Anas G\stanza_resources\resources.json
2025-10-18 11:46:22 WARNING: Language id package default expects mwt, which has been added
2025-10-18 11:46:22 WARNING: Language id package default expects mwt, which has been added
2025-10-18 11:46:22 INFO: Loading these models for language: id (Indonesian):
| Processor | Package    |
--------------------------
| tokenize  | gsd        |
| mwt       | gsd        |
| pos       | gsd_charlm |

2025-10-18 11:46:22 INFO: Using device: cpu
2025-10-18 11:46:22 INFO: Loading: tokenize
2025-10-18 11:46:22 INFO: Loading these models for language: id (Indonesian):
| Processor | Package    |
--------------------------
| tokenize  | gsd        |
| mwt       | gsd        |
| pos       | gsd_charlm |

2025-10-18 11:46:22 INFO: Using device: cpu
2025-10-18 11:46:22 INFO: Loading: tokenize
2025-10-18 11:46:22 INFO: Loading: mwt
2025-10-18 11:46:22 INFO: Loading: mwt
2025-10-18 11:46:22 INFO: Loading: pos
2025

=== DEBUG INFO ===
Total rows in df_tj: 3016

Unique values in 'sentiment_rating' column:
sentiment_rating
Positive    2379
Negative     535
Neutral      102
Name: count, dtype: int64

Data types: object

After normalization:
sentiment_rating
positive    2379
negative     535
neutral      102
Name: count, dtype: int64

Positive sentiment rows: 2379
Negative sentiment rows: 535
Positive rows with POS tags: 2379
Negative rows with POS tags: 535

Top POS Positive results: 3 entries
Top POS Negative results: 3 entries
Total unique word-POS pairs: 3691

File 'c:\Users\Anas G\OneDrive - Institut Teknologi Sepuluh Nopember\Desktop\ANAS BELONGINGS\KULIAH\Semester 7\PBA\Tugas1A\Week5 (POS)\hasil_pos_tag_results_transjakarta.xlsx' berhasil dibuat dengan 3 sheet:
- Top POS Positif
- Top POS Negatif
- All Words

File 'c:\Users\Anas G\OneDrive - Institut Teknologi Sepuluh Nopember\Desktop\ANAS BELONGINGS\KULIAH\Semester 7\PBA\Tugas1A\Week5 (POS)\hasil_pos_tag_results_transjakarta.xlsx' berhasil dib

In [10]:
df_tj['POS']

0                                          [(nan, PROPN)]
1       [(aplikasi, NOUN), (lot, NOUN), (gk, X), (resp...
2                                          [(coba, VERB)]
3       [(aplikasi, NOUN), (mudah, ADJ), (kalo, X), (o...
4                                          [(nan, PROPN)]
                              ...                        
3011                                      [(mantap, ADJ)]
3012       [(crash, NOUN), (pilih, VERB), (profil, NOUN)]
3013    [(aplikasi, NOUN), (kocag, NOUN), (becu, NOUN)...
3014    [(aplikasi, NOUN), (server, NOUN), (eror, NOUN...
3015                                      [(wihh, PROPN)]
Name: POS, Length: 3016, dtype: object